# Ligands and parameter files

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/kdidi/sphinx-docs/docs/tutorial/07_ligand_and_params.ipynb)

> **Prerequisite.** This notebook reuses CIF input and AtomWorks selections from [Working with TMol](01_working_with_tmol.ipynb) and block-pair accounting from [Scoring and Analysis](03_scoring_and_analysis.ipynb).

## Goals

This tutorial uses the checked-in ADA protein–ligand fixtures, with no downloads and no stochastic ligand generation. You will:

- compare Rosetta's line-oriented `.params` representation with TMol's versioned `.tmol` YAML bundle and canonical split YAML databases;
- write both formats from one existing `LigandPreparation`;
- demonstrate the deliberately partial, lossy Rosetta reader;
- prepare a fresh ligand from authoritative MOL2 chemistry;
- inject `.tmol` data into an immutable `ParameterDatabase` and reuse one ligand-aware build context;
- select a ligand and pocket from a Biotite `AtomArray`;
- score the ligand–protein block-pair interaction; and
- repack and minimize a compact ligand pocket with before/after metrics.

## Setup

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "kdidi/sphinx-docs/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(
        [
            "tmol/tests/data/protein_ligand_test/ada.xtal-lig.mmff94.tmol",
            "tmol/tests/data/protein_ligand_test/ada.tmol.nomin.cif",
            "tmol/tests/data/ligand_test/ligand_ground_truth/mol2/ampc_1.mol2",
        ]
    )

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path
import tempfile

import attrs
import biotite.structure as struc
import biotite.structure.io
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

import tmol
from tmol.database import ParameterDatabase
from tmol.io.pose_stack_from_biotite import (
    build_context_from_biotite,
    pose_stack_from_biotite,
)
from tmol.ligand.detect import nonstandard_residue_info_from_mol2
from tmol.ligand.params_file import inject_params_file, load_params_file
from tmol.ligand.params_io import read_params_file, write_params_file
from tmol.ligand.preparation import prepare_single_ligand
from tmol.score import beta2016_score_function
from tmol.score.score_utils import calculate_block_pair_ddg

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LIGAND_RES_NAME = "LG1"


def show_table(frame):
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


def ligand_block_mask(pose_stack):
    mask = torch.zeros_like(pose_stack.block_type_ind, dtype=torch.bool)
    for pose_i in range(pose_stack.n_poses):
        for block_i in range(pose_stack.max_n_blocks):
            type_i = int(pose_stack.block_type_ind[pose_i, block_i])
            if type_i < 0:
                continue
            block_type = pose_stack.packed_block_types.active_block_types[type_i]
            mask[pose_i, block_i] = block_type.name3 == LIGAND_RES_NAME
    if not bool(mask.any()):
        raise RuntimeError(f"No {LIGAND_RES_NAME} ligand block was built")
    return mask

## Two parameter representations

Prefer mmCIF/CIF for structure input, especially for protein–ligand systems: CIF chemical-component and bond tables can preserve connectivity and bond order that PDB coordinate records do not reliably encode. Retain MOL2 or prepared `.tmol` chemistry when it is the authoritative ligand source; a coordinate-only conversion cannot recover missing bond orders.

Rosetta `.params` is a line-oriented, normally one-residue format. Ligand records commonly include `ATOM`, `BOND`/`BOND_TYPE`, `CHI`, `PROTON_CHI`, `NBR_ATOM`, and `ICOOR_INTERNAL`.

TMol's portable `.tmol` file is versioned YAML with three top-level payloads:

- `chemical`: residue atoms, bonds, internal coordinates, torsions, and properties;
- `elec`: per-atom partial charges; and
- `cartbonded`: residue-specific bonded parameters.

The same schemas are split across TMol's canonical database files at `tmol/database/default/chemical/chemical.yaml`, `tmol/database/default/scoring/elec.yaml`, and `tmol/database/default/scoring/cartbonded.yaml`. A portable `.tmol` file bundles ligand additions to those three domains; `chemical.yaml` alone is not a complete scoring parameter set.

In [ ]:
repo_root = Path.cwd()
if not (
    repo_root / "tmol/tests/data/protein_ligand_test/ada.tmol.nomin.cif"
).exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
data_dir = repo_root / "tmol" / "tests" / "data" / "protein_ligand_test"
reference_tmol = data_dir / "ada.xtal-lig.mmff94.tmol"
complex_cif = data_dir / "ada.tmol.nomin.cif"
ligand_mol2 = (
    repo_root
    / "tmol"
    / "tests"
    / "data"
    / "ligand_test"
    / "ligand_ground_truth"
    / "mol2"
    / "ampc_1.mol2"
)

# Loading the checked-in .tmol yields the same LigandPreparation abstraction
# produced by MOL2/CIF/SMILES preparation, without OpenBabel or network access.
preparation = load_params_file(reference_tmol)[0]
work_dir = Path(tempfile.mkdtemp(prefix="tmol-ligand-tutorial-"))
rosetta_path = work_dir / "LG1.params"
tmol_path = work_dir / "LG1.tmol"
write_params_file(preparation, rosetta_path, format="rosetta")
write_params_file(preparation, tmol_path, format="tmol")

print("Wrote:", rosetta_path)
print("Wrote:", tmol_path)

In [ ]:
rosetta_lines = rosetta_path.read_text().splitlines()
tmol_document = yaml.safe_load(tmol_path.read_text())

print("Rosetta .params excerpt")
print("\n".join(rosetta_lines[:18]))
print("\nTMol .tmol excerpt")
print("\n".join(tmol_path.read_text().splitlines()[:18]))

records = ["ATOM", "BOND", "BOND_TYPE", "CHI", "PROTON_CHI", "NBR_ATOM", "ICOOR_INTERNAL"]
record_counts = {
    record: sum(line.startswith(record + " ") for line in rosetta_lines)
    for record in records
}
format_comparison = pd.DataFrame(
    [
        {"representation": "Rosetta .params", "top-level/records": ", ".join(f"{k}:{v}" for k, v in record_counts.items())},
        {"representation": "TMol .tmol", "top-level/records": ", ".join(tmol_document.keys())},
        {"representation": "Canonical split YAML", "top-level/records": "chemical.yaml + elec.yaml + cartbonded.yaml"},
    ]
)
show_table(format_comparison)

## The Rosetta reader is intentionally lossy

`read_params_file()` supports the chemistry/topology subset needed by current tests: names, `ATOM`, `BOND`/`BOND_TYPE`, `CHI`, `PROTON_CHI`, `NBR_ATOM`, and `ICOOR_INTERNAL`. It ignores other Rosetta records. A `RawResidueType` reconstructed from `.params` does not contain the separate TMol electrostatic-charge map or residue-specific cartbonded parameter tables. Therefore `.params → RawResidueType` is not a lossless `.tmol` conversion.

**Expected observation.** Topology counts should survive this file's write/read path, while charge and cartbonded rows remain explicitly absent. A matching atom count alone does not establish scoring equivalence.

In [ ]:
partial_residue_type = read_params_file(rosetta_path)
source_residue_type = preparation.residue_type
count_pairs = [
    ("atoms", len(source_residue_type.atoms), len(partial_residue_type.atoms)),
    ("bonds", len(source_residue_type.bonds), len(partial_residue_type.bonds)),
    ("torsions / chi declarations", len(source_residue_type.torsions), len(partial_residue_type.torsions)),
    ("internal coordinates", len(source_residue_type.icoors), len(partial_residue_type.icoors)),
]
lossiness_rows = [
    {
        "field": field,
        "source_count": source_count,
        "reader_count": reader_count,
        "count_preserved": source_count == reader_count,
    }
    for field, source_count, reader_count in count_pairs
]
lossiness_rows.extend(
    [
        {
            "field": "partial-charge map",
            "source_count": len(preparation.partial_charges),
            "reader_count": 0,
            "count_preserved": False,
        },
        {
            "field": "cartbonded parameter groups",
            "source_count": len(attrs.fields(type(preparation.cartbonded_params))),
            "reader_count": 0,
            "count_preserved": False,
        },
    ]
)
lossiness = pd.DataFrame(lossiness_rows)
show_table(lossiness)
print("Count equality checks structure, not full parameter-value equivalence.")

## Prepare a ligand from MOL2

The portable `.tmol` above is the reproducible archival input. To demonstrate the preparation path itself, the next cell reads the checked-in MOL2—with explicit hydrogens, Tripos bond types, and MMFF94 partial charges—and constructs a fresh `LigandPreparation`. No network request or stochastic conformer generation is involved.

This is the important boundary: TMol does not infer trustworthy bond orders or charges from protein–ligand Cartesian coordinates. Preparation starts from authoritative ligand chemistry, then produces the residue type, charge map, and cartbonded parameters needed for scoring.

In [ ]:
mol2_info = nonstandard_residue_info_from_mol2(
    ligand_mol2, res_name=LIGAND_RES_NAME
)
generated_preparation = prepare_single_ligand(mol2_info)
generated_tmol_path = work_dir / "LG1.generated.tmol"
write_params_file(generated_preparation, generated_tmol_path, format="tmol")

preparation_frame = pd.DataFrame(
    [
        {
            "source": "checked-in .tmol",
            "atoms": len(preparation.residue_type.atoms),
            "bonds": len(preparation.residue_type.bonds),
            "torsions": len(preparation.residue_type.torsions),
            "partial_charges": len(preparation.partial_charges),
        },
        {
            "source": "fresh MOL2 preparation",
            "atoms": len(generated_preparation.residue_type.atoms),
            "bonds": len(generated_preparation.residue_type.bonds),
            "torsions": len(generated_preparation.residue_type.torsions),
            "partial_charges": len(generated_preparation.partial_charges),
        },
    ]
)
show_table(preparation_frame)
print("Generated portable parameters:", generated_tmol_path)

## Inject parameters and build one reusable context

`ParameterDatabase` values are immutable: injection returns a new database and leaves the default database unchanged. We then derive one structure-independent `BiotitePoseBuildContext` from that extended database and construct the pose with `context=context`. This guarantees that pose building and scoring use the same ligand definitions, charges, and cartbonded parameters.

**Expected observation.** The extended database has one additional ligand residue, the original database is unchanged, and the build context holds the exact extended database object. If pose construction reports an unknown `LG1`, the parameter file and build context were not threaded through the same workflow.

In [ ]:
base_database = ParameterDatabase.get_default()
extended_database = inject_params_file(base_database, tmol_path)

complex_array = biotite.structure.io.load_structure(
    str(complex_cif), model=1, include_bonds=True
)
if isinstance(complex_array, struc.AtomArrayStack):
    complex_array = complex_array[0]

pose_diagnostics = StringIO()
try:
    with redirect_stdout(pose_diagnostics), redirect_stderr(pose_diagnostics):
        context = build_context_from_biotite(
            complex_array,
            device,
            param_db=extended_database,
            prepare_ligands=False,
        )
        pose_stack = pose_stack_from_biotite(
            complex_array,
            device,
            context=context,
            no_optH=True,
        )
except Exception:
    print(pose_diagnostics.getvalue())
    raise
score_function = beta2016_score_function(
    device, param_db=context.parameter_database
)

print("default residues:", len(base_database.chemical.residues))
print("extended residues:", len(extended_database.chemical.residues))
print("context reuses extended database:", context.parameter_database is extended_database)

## AtomArray ligand and pocket queries

When docs-only AtomWorks is installed, its Biotite patch provides `AtomArray.mask(expression)` and `AtomArray.query(expression)`. The guarded cell uses those methods for the ligand and falls back to the equivalent NumPy annotation mask. The pocket is a spatial AtomArray selection: non-ligand atoms within 6 Å of any ligand heavy atom. Selection stays in the rich input representation before conversion to TMol block masks.

**Expected observation.** The ligand card should isolate one `LG1` residue; the pocket card should include nearby protein atoms but exclude the ligand. An empty ligand selection usually means the CIF residue name and prepared `.tmol` `name3` disagree.

In [ ]:
ligand_expression = f"res_name == '{LIGAND_RES_NAME}'"
ligand_query = complex_array.res_name == LIGAND_RES_NAME
ligand_atoms = complex_array[ligand_query]
atomworks_available = False
try:
    from atomworks.biotite_patch import monkey_patch_biotite

    monkey_patch_biotite()
    ligand_query = complex_array.mask(ligand_expression)
    ligand_atoms = complex_array.query(ligand_expression)
    atomworks_available = True
except ImportError:
    print("AtomWorks is optional; using the equivalent NumPy ligand mask.")

ligand_heavy_query = ligand_query & (complex_array.element != "H")
ligand_heavy_coords = complex_array.coord[ligand_heavy_query]
all_to_ligand = complex_array.coord[:, None, :] - ligand_heavy_coords[None, :, :]
nearest_ligand_distance = np.linalg.norm(all_to_ligand, axis=-1).min(axis=1)
pocket_query = (~ligand_query) & (nearest_ligand_distance <= 6.0)
pocket_atoms = complex_array[pocket_query]
pocket_residues = sorted(
    {
        (str(chain), int(resid), str(name))
        for chain, resid, name in zip(
            pocket_atoms.chain_id, pocket_atoms.res_id, pocket_atoms.res_name
        )
    }
)
selection_table = pd.DataFrame(
    [
        {"selection": "ligand", "atoms": ligand_atoms.array_length(), "residues": 1},
        {"selection": "6 Å pocket", "atoms": pocket_atoms.array_length(), "residues": len(pocket_residues)},
    ]
)
show_table(selection_table)
display(
    tmol.selection_gallery(
        complex_array,
        {"ligand": ligand_query, "6 Å pocket": pocket_query},
    )
)
pocket_residues[:10]

## Score the ligand–protein interaction

The block-pair scoring module returns a weighted matrix with shape `[n_poses, max_n_blocks, max_n_blocks]`. We select the `LG1` block from the ligand-aware pose and sum both ligand→protein and protein→ligand orientations, matching TMol's block-pair interaction convention. This is an interaction-energy diagnostic, not a physical binding free energy.

**Expected observation.** The interaction value must be finite and the viewer should center on `LG1`. Its sign and magnitude are model-dependent; do not interpret one raw block-pair sum as affinity or ΔG. The viewer converts coordinates to PDB text for display, so ligand bonds are a geometry-oriented visualization—the CIF/`.tmol` data remain authoritative for chemistry.

In [ ]:
ligand_blocks = ligand_block_mask(pose_stack)
real_blocks = pose_stack.block_type_ind >= 0
protein_blocks = real_blocks & ~ligand_blocks

block_pair_scorer = score_function.render_block_pair_scoring_module(pose_stack)
block_pair_scores = block_pair_scorer(pose_stack.coords)
interaction_score = (
    block_pair_scores[0]
    * ligand_blocks[0, :, None]
    * protein_blocks[0, None, :]
).sum() + (
    block_pair_scores[0]
    * protein_blocks[0, :, None]
    * ligand_blocks[0, None, :]
).sum()

show_table(
    pd.DataFrame(
        [
            {
                "metric": "weighted ligand–protein block-pair interaction",
                "value": float(interaction_score.detach().cpu()),
            }
        ]
    )
)

try:
    viewer = tmol.view(pose_stack, zoom_to={"resn": LIGAND_RES_NAME})
    viewer.setStyle(
        {"resn": LIGAND_RES_NAME},
        {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}},
    )
    viewer.show()
except ImportError:
    print("Install py3Dmol for the interactive ligand-pocket view.")

## Refine the ligand pocket

Here the ligand plus complete protein residues with any heavy atom within 4.5 Å are rebuilt as a compact pocket pose. Despite its historical name, `calculate_block_pair_ddg()` returns the weighted cross-mask block-pair interaction from one complex. With `pack=True` it repacks the masked/adjacent blocks; with `minimize=True` it Cartesian-minimizes all ligand atoms and nearby protein side-chain atoms before scoring. It does not construct or subtract an unbound state.

The table therefore labels the result as a ligand–protein interaction score. The returned pose, displacement metric, and structure switcher expose the structural operation that preceded the measurement.

In [ ]:
heavy_non_ligand = (~ligand_query) & (complex_array.element != "H")
within_refinement_radius = heavy_non_ligand & (nearest_ligand_distance <= 4.5)
refinement_residues = {
    (str(chain), int(resid), str(name))
    for chain, resid, name in zip(
        complex_array.chain_id[within_refinement_radius],
        complex_array.res_id[within_refinement_radius],
        complex_array.res_name[within_refinement_radius],
    )
}
protein_refinement_query = np.zeros(complex_array.array_length(), dtype=bool)
for chain, resid, name in refinement_residues:
    protein_refinement_query |= (
        (complex_array.chain_id == chain)
        & (complex_array.res_id == resid)
        & (complex_array.res_name == name)
    )
refinement_array = complex_array[ligand_query | protein_refinement_query]
refinement_diagnostics = StringIO()
try:
    with redirect_stdout(refinement_diagnostics), redirect_stderr(refinement_diagnostics):
        refinement_context = build_context_from_biotite(
            refinement_array,
            device,
            param_db=extended_database,
            prepare_ligands=False,
        )
        refinement_pose = pose_stack_from_biotite(
            refinement_array,
            device,
            context=refinement_context,
            no_optH=True,
        )
except Exception:
    print(refinement_diagnostics.getvalue())
    raise
refinement_sfxn = beta2016_score_function(
    device, param_db=refinement_context.parameter_database
)
refinement_ligand_mask = ligand_block_mask(refinement_pose)

torch.manual_seed(SEED)
interaction_before = calculate_block_pair_ddg(
    refinement_pose,
    refinement_ligand_mask,
    sfxn=refinement_sfxn,
    minimize=False,
    pack=False,
    database=extended_database,
)
interaction_after, refined_pose = calculate_block_pair_ddg(
    refinement_pose,
    refinement_ligand_mask,
    sfxn=refinement_sfxn,
    minimize=True,
    pack=True,
    database=extended_database,
    return_pose_stack=True,
)
refinement_total_scorer = refinement_sfxn.render_whole_pose_scoring_module(
    refinement_pose
)
refined_total_scorer = refinement_sfxn.render_whole_pose_scoring_module(refined_pose)
total_before = float(refinement_total_scorer(refinement_pose.coords).detach().cpu()[0])
total_after = float(refined_total_scorer(refined_pose.coords).detach().cpu()[0])
real_atoms = refinement_pose.real_atoms & refined_pose.real_atoms
coordinate_delta = refined_pose.coords[real_atoms] - refinement_pose.coords[real_atoms]
refinement_rms = float(
    torch.sqrt(torch.mean(torch.sum(coordinate_delta.square(), dim=-1))).detach().cpu()
)

refinement_frame = pd.DataFrame(
    [
        {
            "stage": "input pocket",
            "total_score": total_before,
            "ligand_protein_interaction": float(interaction_before.detach().cpu()[0]),
            "RMS_displacement_A": 0.0,
        },
        {
            "stage": "repacked + minimized",
            "total_score": total_after,
            "ligand_protein_interaction": float(interaction_after.detach().cpu()[0]),
            "RMS_displacement_A": refinement_rms,
        },
    ]
)
show_table(refinement_frame)
display(
    tmol.switchable_view(
        {"input pocket": refinement_pose, "refined pocket": refined_pose},
        notes={
            "input pocket": f"interaction {float(interaction_before.detach().cpu()[0]):.3f}",
            "refined pocket": (
                f"interaction {float(interaction_after.detach().cpu()[0]):.3f}; "
                f"RMS motion {refinement_rms:.3f} Å"
            ),
        },
    )
)

## Rosetta comparison

Rosetta's `.params` format combines residue topology, atom types, charges, internal coordinates, and sampling declarations in line-oriented records. TMol's `.tmol` format mirrors its `ParameterDatabase`: chemistry, electrostatics, and cartbonded terms remain distinct but travel in one versioned YAML document. That separation also matches the canonical split YAML databases.

TMol can write semantically related `.params` and `.tmol` files from one `LigandPreparation`, but the formats are not equivalent. The partial Rosetta reader cannot reconstruct all charge/cartbonded information from arbitrary `.params` files. Preserve the original `.tmol` or source chemistry when TMol scoring fidelity matters.

Rosetta's ligand workflows include global docking and broader residue-type machinery. This tutorial demonstrates preparation, parameter registration, local pocket repacking/minimization, and interaction diagnostics, but it does not implement global docking or claim protocol parity with RosettaLigand or GALigandDock.

## Limitations

- TMol does not promise a ligand-docking protocol, covalent-ligand modeling, or general metal coordination.
- Metal-containing and cross-residue/covalent ligands are currently unsupported by the automatic preparation path.
- PDB does not reliably encode ligand bond order; prefer CIF, MOL2, or prepared `.tmol` chemistry.
- MOL2/SMILES preparation requires the optional chemistry stack; the Colab bootstrap installs it, while the pinned `.tmol` remains the reproducible fallback artifact.
- The Rosetta reader supports only a documented subset and silently ignores other records; conversion is lossy.
- Pocket refinement is local repacking/minimization, not a ligand pose search or docking protocol.
- The block-pair value is a score-function interaction proxy, not a docking score, ΔG, or experimental affinity.

## Exercises

1. Compare atom names, charges, and cartbonded values—not only counts—between the fresh MOL2 preparation and pinned `.tmol` artifact.
2. Count which `CHI` and `PROTON_CHI` declarations survive a Rosetta write/read round trip.
3. Change the AtomArray pocket cutoff and quantify its effect on runtime and refined interaction score.
4. Render `sum_terms=False` block-pair scores and rank score-term contributions before and after refinement.
5. Repeat pocket refinement with several packer seeds and report the score and structural spread without calling it a binding free energy.

## References

- [PyRosetta Workshop Appendix B: residue parameter files](https://graylab.jhu.edu/pyrosetta/downloads/documentation/pyrosetta4_online_format/PyRosetta4_Workshops_Appendix_B.pdf)
- [Rosetta residue `.params` reference](https://docs.rosettacommons.org/docs/latest/rosetta_basics/file_types/Residue-Params-file)
- [Rosetta ligand-preparation tutorial](https://rosettacommons.org/demos/latest/tutorials/prepare_ligand/prepare_ligand_tutorial)
- [RosettaLigand workshop slides](https://meilerlab.org/wp-content/uploads/2025/11/ligand_docking_presentation.pdf) and [2025 ligand-docking exercise](https://meilerlab.org/wp-content/uploads/2025/03/ligand_docking_tutorial.pdf)
- [Rosetta `molfile_to_params.py`](https://github.com/RosettaCommons/rosetta/blob/main/source/scripts/python/public/molfile_to_params.py)
- [PyRosettaCluster ligand-parameter workflow](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.10-PyRosettaCluster-Ligand-params.ipynb)
- [TMol ligand user guide](../user_guide/ligands.md)